In [ ]:
# Construct Diffuser

def build_diffuser(n):
    s = QuantumRegister(n, "s")
    anc = AncillaRegister(max(1, n - 1), "anc_d")
    qc = QuantumCircuit(s, anc, name="Diffuser")

    qc.h(s)
    qc.x(s)

    qc.h(s[n - 1])
    mcx_generic(qc, list(s[:-1]), s[n - 1])
    qc.h(s[n - 1])

    qc.x(s)
    qc.h(s)

    return qc

In [ ]:
#applied Grover algorithm(repeat oracle and diffuser)
def subset_sum_quantum(xs, t, iters=None):
    n = len(xs)
    max_sum = sum(xs)
    m = max(1, math.ceil(math.log2(max_sum + 1)))

    if iters is None:
        iters = max(1, int((math.pi / 4) * math.sqrt(2 ** n)))

    subset = QuantumRegister(n, "s")
    acc = QuantumRegister(m, "a")
    anc_or = AncillaRegister(max(1, m - 1), "anc_or")
    ph = QuantumRegister(1, "ph")
    anc_d = AncillaRegister(max(1, n - 1), "anc_d")

    qc = QuantumCircuit(subset, acc, anc_or, ph, anc_d)

    qc.h(subset)

    oracle = build_subset_sum_phase_oracle(xs, t, m).to_gate()
    diffuser = build_diffuser(n).to_gate()

    for _ in range(iters):
        qc.append(oracle, list(subset) + list(acc) + list(anc_or) + list(ph))
        qc.append(diffuser, list(subset) + list(anc_d))

    return qc

In [ ]:
#applied alorithm and chose the high probability result
def sample_solutions(xs, t, shots=500, iters=2):
    from qiskit.primitives import StatevectorSampler

    qc = subset_sum_quantum(xs, t, iters=iters)

    n = len(xs)
    c = ClassicalRegister(n, "c_s")
    qc.add_register(c)

    # measure only subset register (first qreg)
    qc.measure(qc.qregs[0], c)

    sampler = StatevectorSampler()
    result = sampler.run([qc], shots=shots).result()

    counts = result[0].data.c_s.get_counts()

    best = max(counts, key=counts.get)

    # Qiskit prints MSB..LSB; reverse to map to s_0..s_{n-1}
    s_bits = best[::-1]
    I = {i for i, b in enumerate(s_bits) if b == "1"}

    return counts, best, s_bits, I